<a href="https://colab.research.google.com/github/makenziewiggins/The-Urban-Equity-Sunshine-Warehouse/blob/main/Transit_Equit_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚌 Transit Equity Data Warehouse — ETL Pipeline
### Makenzie Wiggins | Data Warehousing Final Project
**Project:** A BI Framework for Auditing Transit Equity: Central Florida

---
## Pipeline Architecture
```
Raw Data (Bronze Layer)
       │
       ▼
ETL Transformations (Silver Layer)
       │
       ▼
Star Schema (Gold Layer — loaded separately)
```
**Data Sources:**
- LYNX GTFS (Transit schedules & stop locations)
- Orange County GIS (Neighborhood boundaries)
- US Census ACS 5-Year 2024 (Demographics)
- TIGER/Line 2024 Census Tracts (Spatial boundaries)


##  1 — Install & Import Libraries

In [2]:
# Install required libraries (run once in Colab)
!pip install geopandas folium mapclassify --quiet

# Core libraries
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import zipfile
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


## 2 - Mount Google Drive & Set File Paths

In [9]:
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ UPDATE THIS PATH to match where your folder lives in Google Drive
BASE_PATH = '/content/drive/MyDrive/Transit Equity Project/'  # <-- change this if needed

# --- GTFS File Paths ---
GTFS_STOPS      = os.path.join(BASE_PATH, 'stops.txt')
GTFS_STOP_TIMES = os.path.join(BASE_PATH, 'stop_times.txt')
GTFS_ROUTES     = os.path.join(BASE_PATH, 'routes.txt')
GTFS_TRIPS      = os.path.join(BASE_PATH, 'trips.txt')
GTFS_CALENDAR   = os.path.join(BASE_PATH, 'calendar.txt')

# --- Geospatial File Paths ---
OC_ZONING_GEOJSON   = os.path.join(BASE_PATH, 'Orange_County_Zoning.geojson')
TRACT_SHAPEFILE     = os.path.join(BASE_PATH, 'tl_2024_12_tract.shp')

# --- Census Data Paths ---
CENSUS_S0801  = os.path.join(BASE_PATH, 'ACSST5Y2024.S0801-Data.csv')   # Commuting
CENSUS_S1901  = os.path.join(BASE_PATH, 'ACSST5Y2024.S1901-Data.csv')   # Income
CENSUS_S1701  = os.path.join(BASE_PATH, 'ACSST5Y2024.S1701-Data.csv')   # Poverty
CENSUS_B08201 = os.path.join(BASE_PATH, 'ACSDT5Y2024.B08201-2026-04-14T221030.csv')  # Vehicle availability
CENSUS_B02001 = os.path.join(BASE_PATH, 'ACSDT5Y2024.B02001-2026-04-14T220735.csv')  # Race
CENSUS_B03002 = os.path.join(BASE_PATH, 'ACSDT5Y2024.B03002-2026-04-14T220905.csv')  # Hispanic/Latino

print('✅ File paths configured')
print(f'   Base path: {BASE_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ File paths configured
   Base path: /content/drive/MyDrive/Transit Equity Project/


## 3 - BRONZE LAYER: Load Raw GTS Data

In [4]:
# ============================================================
# BRONZE LAYER — Raw ingestion, no transformations applied
# Goal: Load exactly as-is and document what I received
# ============================================================

print('=' * 55)
print('BRONZE LAYER — GTFS Raw Data Ingestion')
print('=' * 55)

# Load each GTFS file
bronze_stops      = pd.read_csv(GTFS_STOPS)
bronze_stop_times = pd.read_csv(GTFS_STOP_TIMES)
bronze_routes     = pd.read_csv(GTFS_ROUTES)
bronze_trips      = pd.read_csv(GTFS_TRIPS)
bronze_calendar   = pd.read_csv(GTFS_CALENDAR)

# Data profiling summary
gtfs_files = {
    'stops':      bronze_stops,
    'stop_times': bronze_stop_times,
    'routes':     bronze_routes,
    'trips':      bronze_trips,
    'calendar':   bronze_calendar
}

print('\n📊 GTFS Data Profiling Summary:')
print(f'{"File":<15} {"Rows":>10} {"Columns":>10} {"Null Values":>15}')
print('-' * 55)
for name, df in gtfs_files.items():
    print(f'{name:<15} {len(df):>10,} {len(df.columns):>10} {df.isnull().sum().sum():>15,}')

print('\n✅ GTFS Bronze layer loaded')

BRONZE LAYER — GTFS Raw Data Ingestion

📊 GTFS Data Profiling Summary:
File                  Rows    Columns     Null Values
-------------------------------------------------------
stops                3,764          6               0
stop_times         726,286          5       1,302,696
routes                  62          4               0
trips               15,353          5               0
calendar                16         10               0

✅ GTFS Bronze layer loaded


In [5]:
# Quick peek at key files
print('--- stops.txt sample ---')
display(bronze_stops.head(3))

print('\n--- routes.txt sample ---')
display(bronze_routes.head(3))

print('\n--- stop_times.txt sample ---')
display(bronze_stop_times.head(3))

--- stops.txt sample ---


,stop_id,stop_name,stop_lat,stop_lon,location_type,wheelchair_boarding
0,175,ROSEMONT SUPERSTOP,28.61,-81.43,0,1
1,214,KOA ST & MONTEREY RD,28.16,-81.48,0,2
2,555,ST CLOUD WALMART,28.25,-81.31,0,1



--- routes.txt sample ---


,route_id,route_short_name,route_long_name,route_type
0,16828,03,LAKE MARGARET DR.,3
1,16829,06,DIXIE BELLE DRIVE/BUMBY AVE,3
2,16830,07,S. ORANGE AVE/FLORIDA MALL,3



--- stop_times.txt sample ---


,trip_id,arrival_time,departure_time,stop_id,stop_sequence
0,2114898,05:25:00,05:25:00,4774,1
1,2114898,NaN,NaN,4057,2
2,2114898,NaN,NaN,4428,3


## 4 - BRONZE LAYER: Load Raw Geospatial Data

In [6]:
print('=' * 55)
print('BRONZE LAYER — Geospatial Raw Data Ingestion')
print('=' * 55)

# Load Orange County Zoning/Neighborhood boundaries
bronze_oc_zones = gpd.read_file(OC_ZONING_GEOJSON)

# Load Census Tract shapefile
bronze_tracts = gpd.read_file(TRACT_SHAPEFILE)

print(f'\n📊 Geospatial Data Profiling:')
print(f'  OC Zoning GeoJSON  — {len(bronze_oc_zones):,} features | CRS: {bronze_oc_zones.crs}')
print(f'  Census Tracts      — {len(bronze_tracts):,} tracts   | CRS: {bronze_tracts.crs}')

print('\n--- OC Zoning Columns ---')
print(bronze_oc_zones.columns.tolist())

print('\n--- Census Tract Columns ---')
print(bronze_tracts.columns.tolist())

print('\n✅ Geospatial Bronze layer loaded')

BRONZE LAYER — Geospatial Raw Data Ingestion

📊 Geospatial Data Profiling:
  OC Zoning GeoJSON  — 17,706 features | CRS: EPSG:4326
  Census Tracts      — 5,160 tracts   | CRS: EPSG:4269

--- OC Zoning Columns ---
['OBJECTID', 'ZONING', 'P_Z_DATE', 'PD_NAME', 'COMMENTS', 'BCC_DATE', 'MAINT_DATE', 'ZONINGOLD', 'ZONETYPE', 'JURISDICTION', 'P_Z_DATE_OLD', 'HEARING_NUM', 'SHAPE_Length', 'SHAPE_Area', 'geometry']

--- Census Tract Columns ---
['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']

✅ Geospatial Bronze layer loaded


## 5 - BRONZE LAYER: Load Raw Census Data

In [10]:
print('=' * 55)
print('BRONZE LAYER — Census ACS 2024 Raw Data Ingestion')
print('=' * 55)

# Census CSVs have 2 header rows — row 0 = codes, row 1 = labels
# We skip row 1 (index 1) to keep the machine-readable codes as headers
def load_census(filepath, label):
    df = pd.read_csv(filepath, skiprows=[1], low_memory=False)
    print(f'  {label:<20} — {len(df):,} tracts | {len(df.columns):,} columns')
    return df

print('\n📊 Census Data Profiling:')
bronze_s0801  = load_census(CENSUS_S0801,  'S0801 Commuting')
bronze_s1901  = load_census(CENSUS_S1901,  'S1901 Income')
bronze_s1701  = load_census(CENSUS_S1701,  'S1701 Poverty')
bronze_b08201 = load_census(CENSUS_B08201, 'B08201 Vehicles')
bronze_b02001 = load_census(CENSUS_B02001, 'B02001 Race')
bronze_b03002 = load_census(CENSUS_B03002, 'B03002 Hispanic')

print('\n✅ Census Bronze layer loaded')

BRONZE LAYER — Census ACS 2024 Raw Data Ingestion

📊 Census Data Profiling:
  S0801 Commuting      — 267 tracts | 345 columns
  S1901 Income         — 267 tracts | 131 columns
  S1701 Poverty        — 267 tracts | 375 columns
  B08201 Vehicles      — 29 tracts | 535 columns
  B02001 Race          — 9 tracts | 535 columns
  B03002 Hispanic      — 20 tracts | 535 columns

✅ Census Bronze layer loaded


## 6 -  SILVER LAYER: Clean & Transform GTFS Stops

In [11]:
# ============================================================
# SILVER LAYER — Transformation 1: Clean Stops
# Business Rule: Every stop must have valid coordinates
# and a classified service type
# ============================================================

print('=' * 55)
print('SILVER LAYER — Transforming Stops')
print('=' * 55)

silver_stops = bronze_stops.copy()

# --- Data Quality Check 1: Remove stops missing coordinates ---
before = len(silver_stops)
silver_stops = silver_stops.dropna(subset=['stop_lat', 'stop_lon'])
after = len(silver_stops)
print(f'\n🔍 Removed {before - after} stops missing coordinates')

# --- Data Quality Check 2: Validate coordinate range for Orange County FL ---
# Orange County roughly: lat 28.3–28.8, lon -81.7 to -81.0
before = len(silver_stops)
silver_stops = silver_stops[
    (silver_stops['stop_lat'].between(28.3, 28.8)) &
    (silver_stops['stop_lon'].between(-81.7, -81.0))
]
after = len(silver_stops)
print(f'🔍 Removed {before - after} stops outside Orange County bounds')

# --- Type Conversion ---
silver_stops['stop_lat'] = silver_stops['stop_lat'].astype(float)
silver_stops['stop_lon'] = silver_stops['stop_lon'].astype(float)
silver_stops['stop_id']  = silver_stops['stop_id'].astype(str).str.strip()

# --- Business Rule: Classify service type from stop name ---
# LYMMO stops are premium downtown circulators
silver_stops['service_type'] = 'Standard Link'
silver_stops.loc[
    silver_stops['stop_name'].str.contains('LYMMO', case=False, na=False),
    'service_type'
] = 'LYMMO'
silver_stops.loc[
    silver_stops['stop_name'].str.contains('NeighborLink|Neighbor Link', case=False, na=False),
    'service_type'
] = 'NeighborLink'

# --- Amenity Score placeholder (from stops.txt wheelchair_boarding if available) ---
if 'wheelchair_boarding' in silver_stops.columns:
    # 1 = accessible, 2 = not accessible, 0 = unknown
    silver_stops['ada_accessible'] = silver_stops['wheelchair_boarding'].map(
        {1: True, 2: False, 0: None}
    )
else:
    silver_stops['ada_accessible'] = None
    print('⚠️  wheelchair_boarding not in stops.txt — ada_accessible set to None')

# --- Convert to GeoDataFrame for spatial operations ---
silver_stops_geo = gpd.GeoDataFrame(
    silver_stops,
    geometry=gpd.points_from_xy(silver_stops['stop_lon'], silver_stops['stop_lat']),
    crs='EPSG:4326'  # Standard WGS84
)

print(f'\n📊 Silver Stops Summary:')
print(f'   Total stops: {len(silver_stops_geo):,}')
print(f"   Service types:\n{silver_stops_geo['service_type'].value_counts().to_string()}")
print('\n✅ Stops silver layer complete')

SILVER LAYER — Transforming Stops

🔍 Removed 0 stops missing coordinates
🔍 Removed 232 stops outside Orange County bounds

📊 Silver Stops Summary:
   Total stops: 3,532
   Service types:
service_type
Standard Link    3532

✅ Stops silver layer complete


## 7 - SILVER LAYER: Clean & Transform Stop TImes + Calculate Headway

In [12]:
# ============================================================
# SILVER LAYER — Transformation 2: Stop Times & Headway
# Key Challenge: GTFS times can exceed 24:00:00 for overnight
# Business Rule: Headway = minutes between consecutive
# arrivals at the same stop on the same route
# ============================================================

print('=' * 55)
print('SILVER LAYER — Transforming Stop Times & Headway')
print('=' * 55)

silver_stop_times = bronze_stop_times.copy()

# --- Handle GTFS time format (can be >24:00:00 for overnight runs) ---
def gtfs_time_to_minutes(time_str):
    """
    Convert GTFS time string (HH:MM:SS) to total minutes.
    Handles times > 24:00:00 for overnight service.
    Returns NaN for malformed strings.
    """
    try:
        parts = str(time_str).strip().split(':')
        if len(parts) != 3:
            return np.nan
        h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
        return h * 60 + m + s / 60
    except:
        return np.nan

print('\n⏳ Converting GTFS times (this may take a moment for large files)...')
silver_stop_times['arrival_minutes']   = silver_stop_times['arrival_time'].apply(gtfs_time_to_minutes)
silver_stop_times['departure_minutes'] = silver_stop_times['departure_time'].apply(gtfs_time_to_minutes)

# --- Data Quality Check: Drop rows with unparseable times ---
before = len(silver_stop_times)
silver_stop_times = silver_stop_times.dropna(subset=['arrival_minutes'])
after = len(silver_stop_times)
print(f'🔍 Removed {before - after:,} rows with unparseable time values')

# --- Classify Peak vs Off-Peak ---
# Peak: 6-9am and 4-7pm on weekdays
def classify_peak(minutes):
    hour = minutes / 60
    if (6 <= hour < 9) or (16 <= hour < 19):
        return 'Peak'
    elif 0 <= hour < 6 or hour >= 22:
        return 'Late Night'
    else:
        return 'Off-Peak'

silver_stop_times['time_period'] = silver_stop_times['arrival_minutes'].apply(classify_peak)

# --- Calculate Headway ---
# Sort by stop + trip sequence, then compute gap between arrivals
print('\n⏳ Calculating headway per stop...')
silver_stop_times = silver_stop_times.sort_values(['stop_id', 'arrival_minutes'])

silver_stop_times['headway_minutes'] = (
    silver_stop_times
    .groupby('stop_id')['arrival_minutes']
    .diff()
)

# Business Rule: Headway > 120 min is likely a data gap, not real service
before = len(silver_stop_times)
silver_stop_times.loc[silver_stop_times['headway_minutes'] > 120, 'headway_minutes'] = np.nan
print(f'🔍 Flagged unrealistic headway values (>120 min) as null')

# --- Type cleanup ---
silver_stop_times['stop_id']  = silver_stop_times['stop_id'].astype(str).str.strip()
silver_stop_times['trip_id']  = silver_stop_times['trip_id'].astype(str).str.strip()

print(f'\n📊 Headway Summary by Time Period:')
print(silver_stop_times.groupby('time_period')['headway_minutes']
      .agg(['mean','median','count'])
      .round(2)
      .to_string())

print('\n✅ Stop times silver layer complete')

SILVER LAYER — Transforming Stop Times & Headway

⏳ Converting GTFS times (this may take a moment for large files)...
🔍 Removed 651,348 rows with unparseable time values

⏳ Calculating headway per stop...
🔍 Flagged unrealistic headway values (>120 min) as null

📊 Headway Summary by Time Period:
             mean  median  count
time_period                     
Late Night   4.48    0.00   7386
Off-Peak     3.39    0.00  40667
Peak         3.25    0.00  26584

✅ Stop times silver layer complete


## 8 - SILVER LAYER: Clean Routes & Trips

In [15]:
# ============================================================
# SILVER LAYER — Transformation 3: Routes & Trips
# Business Rule: Classify routes into service levels
# ============================================================

print('=' * 55)
print('SILVER LAYER — Transforming Routes & Trips')
print('=' * 55)

silver_routes = bronze_routes.copy()
silver_trips  = bronze_trips.copy()

# --- Routes: Classify service level ---
silver_routes['service_level'] = 'Standard'
silver_routes.loc[
    silver_routes['route_long_name'].str.contains('LYMMO', case=False, na=False) |
    silver_routes['route_short_name'].str.contains('LYMMO', case=False, na=False),
    'service_level'
] = 'LYMMO Premium'
silver_routes.loc[
    silver_routes['route_long_name'].str.contains('NeighborLink|Neighbor', case=False, na=False),
    'service_level'
] = 'NeighborLink'

# --- Fill missing route names ---
silver_routes['route_long_name']  = silver_routes['route_long_name'].fillna('Unknown Route')
silver_routes['route_short_name'] = silver_routes['route_short_name'].fillna(
    silver_routes['route_id'].astype(str)
)

# --- Trips: Clean and join service day info ---
silver_trips['route_id']   = silver_trips['route_id'].astype(str).str.strip()
silver_trips['service_id'] = silver_trips['service_id'].astype(str).str.strip()

# Join calendar to get weekday/weekend flag
silver_calendar = bronze_calendar.copy()
silver_calendar['is_weekday'] = (
    silver_calendar[['monday','tuesday','wednesday','thursday','friday']].max(axis=1) == 1
)
silver_calendar['is_weekend'] = (
    silver_calendar[['saturday','sunday']].max(axis=1) == 1
)

# Force service_id to string in calendar too (was int64, caused merge error)
silver_calendar['service_id'] = silver_calendar['service_id'].astype(str).str.strip()

silver_trips = silver_trips.merge(
    silver_calendar[['service_id','is_weekday','is_weekend']],
    on='service_id',
    how='left'
)

print(f'\n📊 Routes by Service Level:')
print(silver_routes['service_level'].value_counts().to_string())
print(f'\n📊 Trips — Weekday vs Weekend:')
print(f"   Weekday trips: {silver_trips['is_weekday'].sum():,}")
print(f"   Weekend trips: {silver_trips['is_weekend'].sum():,}")
print('\n✅ Routes & trips silver layer complete')

SILVER LAYER — Transforming Routes & Trips

📊 Routes by Service Level:
service_level
Standard         59
LYMMO Premium     3

📊 Trips — Weekday vs Weekend:
   Weekday trips: 6,339
   Weekend trips: 9,014

✅ Routes & trips silver layer complete


## 9 - SILVER LAYER: Spatial Join - Tag Stops with Neighborhoods

In [16]:
# ============================================================
# SILVER LAYER — Transformation 4: Spatial Join
# Business Rule: Every stop gets tagged with its neighborhood
# and an equity_zone flag for Pine Hills / Parramore
# ============================================================

print('=' * 55)
print('SILVER LAYER — Spatial Join: Stops → Neighborhoods')
print('=' * 55)

# Reproject everything to a consistent CRS
# EPSG:4326 = WGS84 (lat/lon), standard for GPS data
silver_oc_zones = bronze_oc_zones.copy()
if silver_oc_zones.crs != 'EPSG:4326':
    silver_oc_zones = silver_oc_zones.to_crs('EPSG:4326')
    print('  Reprojected OC zones to EPSG:4326')

# Print available columns so we can find the neighborhood name field
print('\n📋 OC Zoning GeoJSON columns:')
print(silver_oc_zones.columns.tolist())
display(silver_oc_zones.head(2))

SILVER LAYER — Spatial Join: Stops → Neighborhoods

📋 OC Zoning GeoJSON columns:
['OBJECTID', 'ZONING', 'P_Z_DATE', 'PD_NAME', 'COMMENTS', 'BCC_DATE', 'MAINT_DATE', 'ZONINGOLD', 'ZONETYPE', 'JURISDICTION', 'P_Z_DATE_OLD', 'HEARING_NUM', 'SHAPE_Length', 'SHAPE_Area', 'geometry']


,OBJECTID,ZONING,P_Z_DATE,PD_NAME,COMMENTS,BCC_DATE,MAINT_DATE,ZONINGOLD,ZONETYPE,JURISDICTION,P_Z_DATE_OLD,HEARING_NUM,SHAPE_Length,SHAPE_Area,geometry
0,1,CITY,NaT,,None,NaT,NaT,None,14,Maitland,None,,6944.88,1633233.15,"POLYGON ((-81.38734 28.62505, -81.38735 28.625..."
1,2,P-D,2019-08-15 00:00:00+00:00,Four Corners Plaza PD/LUP,Approved with (10) condition Condition #10 inc...,2019-09-24 00:00:00+00:00,2019-09-26 00:00:00+00:00,A-1,10,Unincorporated,1957-10-07T00:00:00Z,LUP-19-02-063,1575.73,104321.59,"POLYGON ((-81.65539 28.34824, -81.65539 28.348..."


In [22]:
# ⚠️ IMPORTANT: After running the cell above, look at the column names
# printed and update NEIGHBORHOOD_COL below to the correct column name
# that contains neighborhood or zone names in your GeoJSON file.

NEIGHBORHOOD_COL = 'JURISDICTION'

# Perform the spatial join
print(f'\n⏳ Running spatial join (stops inside neighborhood polygons)...')
silver_stops_with_neighborhood = gpd.sjoin(
    silver_stops_geo,
    silver_oc_zones[['geometry', NEIGHBORHOOD_COL]],
    how='left',
    predicate='within'
)

# Rename the neighborhood column for clarity
silver_stops_with_neighborhood = silver_stops_with_neighborhood.rename(
    columns={NEIGHBORHOOD_COL: 'neighborhood_name'}
)

# Fill stops that didn't join to any polygon
silver_stops_with_neighborhood['neighborhood_name'] = (
    silver_stops_with_neighborhood['neighborhood_name'].fillna('Unclassified')
)

# --- Business Rule: Flag Equity Zones ---
# These are the historically underserved neighborhoods in our analysis
EQUITY_ZONES = ['Pine Hills', 'Parramore', 'Richmond Heights', 'Holden Heights']

silver_stops_with_neighborhood['is_equity_zone'] = (
    silver_stops_with_neighborhood['neighborhood_name']
    .str.contains('|'.join(EQUITY_ZONES), case=False, na=False)
)

# --- Summary ---
total = len(silver_stops_with_neighborhood)
matched = (silver_stops_with_neighborhood['neighborhood_name'] != 'Unclassified').sum()
equity = silver_stops_with_neighborhood['is_equity_zone'].sum()

print(f'\n📊 Spatial Join Results:')
print(f'   Total stops:               {total:,}')
print(f'   Stops matched to polygon:  {matched:,} ({matched/total*100:.1f}%)')
print(f'   Stops in equity zones:     {equity:,}')

print(f'\n📊 Top 10 Neighborhoods by Stop Count:')
print(silver_stops_with_neighborhood['neighborhood_name']
      .value_counts().head(10).to_string())

print('\n✅ Spatial join complete')


⏳ Running spatial join (stops inside neighborhood polygons)...

📊 Spatial Join Results:
   Total stops:               3,532
   Stops matched to polygon:  644 (18.2%)
   Stops in equity zones:     0

📊 Top 10 Neighborhoods by Stop Count:
neighborhood_name
Unclassified        2888
Orlando              463
Unincorporated        87
Winter Park           31
Apopka                20
Lake Buena Vista      10
Edgewood               9
Maitland               9
Bay Lake               6
Ocoee                  4

✅ Spatial join complete


## 10 - SILVER LAYER: Clean Census Data

In [24]:
print("S1901 columns:", bronze_s1901.columns[:5].tolist())
print("B08201 columns:", bronze_b08201.columns[:5].tolist())
print("B02001 columns:", bronze_b02001.columns[:5].tolist())

S1901 columns: ['GEO_ID', 'NAME', 'S1901_C01_001E', 'S1901_C01_001M', 'S1901_C01_002E']
B08201 columns: ['Label (Grouping)', 'Census Tract 102.01; Orange County; Florida!!Estimate', 'Census Tract 102.01; Orange County; Florida!!Margin of Error', 'Census Tract 102.02; Orange County; Florida!!Estimate', 'Census Tract 102.02; Orange County; Florida!!Margin of Error']
B02001 columns: ['Label (Grouping)', 'Census Tract 102.01; Orange County; Florida!!Estimate', 'Census Tract 102.01; Orange County; Florida!!Margin of Error', 'Census Tract 102.02; Orange County; Florida!!Estimate', 'Census Tract 102.02; Orange County; Florida!!Margin of Error']


In [25]:
# Transpose B08201 to fix orientation
b08201_fixed = bronze_b08201.copy()
b08201_fixed = b08201_fixed.set_index('Label (Grouping)').T.reset_index()
b08201_fixed.columns.name = None
b08201_fixed = b08201_fixed.rename(columns={'index': 'GEO_ID'})
print(b08201_fixed.columns[:5].tolist())
print(b08201_fixed.head(2))


# Transpose B02001 to fix orientation
b02001_fixed = bronze_b02001.copy()
b02001_fixed = b02001_fixed.set_index('Label (Grouping)').T.reset_index()
b02001_fixed.columns.name = None
b02001_fixed = b02001_fixed.rename(columns={'index': 'GEO_ID'})
print(b02001_fixed.columns[:5].tolist())
print(b02001_fixed.head(2))


# Transpose B03002 to fix orientation
b03002_fixed = bronze_b03002.copy()
b03002_fixed = b03002_fixed.set_index('Label (Grouping)').T.reset_index()
b03002_fixed.columns.name = None
b03002_fixed = b03002_fixed.rename(columns={'index': 'GEO_ID'})
print(b03002_fixed.columns[:5].tolist())
print(b03002_fixed.head(2))

['GEO_ID', '\xa0\xa0\xa0\xa0No vehicle available', '\xa0\xa0\xa0\xa01 vehicle available', '\xa0\xa0\xa0\xa02 vehicles available', '\xa0\xa0\xa0\xa03 vehicles available']
                                              GEO_ID     No vehicle available  \
0  Census Tract 102.01; Orange County; Florida!!E...                       89   
1  Census Tract 102.01; Orange County; Florida!!M...                      ±66   

      1 vehicle available     2 vehicles available     3 vehicles available  \
0                     824                      303                       76   
1                    ±171                      ±61                      ±71   

      4 or more vehicles available     1-person household:  \
0                               10                     769   
1                              ±17                    ±149   

          No vehicle available         1 vehicle available  \
0                           89                         591   
1                          ±66       

In [28]:
print('=' * 55)
print('SILVER LAYER — Transforming Census Data')
print('=' * 55)

# Helper to fix transposed B-code tables
def fix_transposed(df):
    df = df.copy()
    df = df.set_index('Label (Grouping)').T.reset_index()
    df.columns.name = None
    df = df.rename(columns={'index': 'GEO_ID'})
    return df

# Helper to clean GEOID
def clean_geoid(df):
    df = df.copy()
    df['GEOID'] = df['GEO_ID'].astype(str).str.strip()
    # Handle both formats: 'Census Tract 102.01; Orange County; Florida!!Estimate'
    # and standard '1400000US12095000100'
    if df['GEOID'].str.contains('Census Tract', na=False).any():
        # Extract tract number from column header format
        df['GEOID'] = df['GEOID'].str.extract(r'Census Tract ([\d.]+);')[0]
    else:
        df['GEOID'] = df['GEOID'].str[-11:]
    return df

# --- S1901: Income (already in correct format) ---
silver_income = bronze_s1901.copy()
silver_income = clean_geoid(silver_income)
income_col = [c for c in silver_income.columns if 'S1901_C01_012E' in c]
print(f'  Income column found: {income_col}')
if income_col:
    silver_income = silver_income[['GEOID', income_col[0]]].copy()
    silver_income.columns = ['GEOID', 'median_household_income']
    silver_income['median_household_income'] = pd.to_numeric(
        silver_income['median_household_income'], errors='coerce'
    )

# --- S1701: Poverty (already in correct format) ---
silver_poverty = bronze_s1701.copy()
silver_poverty = clean_geoid(silver_poverty)
poverty_col = [c for c in silver_poverty.columns if 'S1701_C03_001E' in c]
print(f'  Poverty column found: {poverty_col}')
if poverty_col:
    silver_poverty = silver_poverty[['GEOID', poverty_col[0]]].copy()
    silver_poverty.columns = ['GEOID', 'pct_poverty']
    silver_poverty['pct_poverty'] = pd.to_numeric(
        silver_poverty['pct_poverty'], errors='coerce'
    )

# --- B08201: Vehicle Availability (transposed — fix first) ---
b08201_fixed = fix_transposed(bronze_b08201)
print(f'\n  B08201 rows after transpose: {len(b08201_fixed)}')
print(f'  B08201 sample columns: {b08201_fixed.columns[:4].tolist()}')

# Find total households and zero vehicle rows
total_col   = [c for c in b08201_fixed.columns if 'Total' in str(c) and 'Estimate' not in str(c)]
zero_v_col  = [c for c in b08201_fixed.columns if 'No vehicle' in str(c) or 'no vehicle' in str(c)]
print(f'  Total col: {total_col}')
print(f'  Zero vehicle col: {zero_v_col}')

silver_vehicles = b08201_fixed[['GEO_ID']].copy()
silver_vehicles['GEOID'] = silver_vehicles['GEO_ID'].astype(str).str.strip()
if total_col and zero_v_col:
    silver_vehicles['total_households']       = pd.to_numeric(b08201_fixed[total_col[0]], errors='coerce')
    silver_vehicles['zero_vehicle_households'] = pd.to_numeric(b08201_fixed[zero_v_col[0]], errors='coerce')
    silver_vehicles['pct_zero_vehicle'] = (
        silver_vehicles['zero_vehicle_households'] / silver_vehicles['total_households'] * 100
    ).round(2)

# --- B02001: Race (transposed — fix first) ---
b02001_fixed = fix_transposed(bronze_b02001)
total_pop_col = [c for c in b02001_fixed.columns if 'Total' in str(c)]
black_col     = [c for c in b02001_fixed.columns if 'Black' in str(c) or 'African American' in str(c)]
print(f'\n  Total pop col: {total_pop_col[:2]}')
print(f'  Black pop col: {black_col[:2]}')

silver_race = b02001_fixed[['GEO_ID']].copy()
silver_race['GEOID'] = silver_race['GEO_ID'].astype(str).str.strip()
if total_pop_col and black_col:
    silver_race['total_population'] = pd.to_numeric(b02001_fixed[total_pop_col[0]], errors='coerce')
    silver_race['black_population']  = pd.to_numeric(b02001_fixed[black_col[0]], errors='coerce')
    silver_race['pct_black'] = (
        silver_race['black_population'] / silver_race['total_population'] * 100
    ).round(2)

# --- Merge all into one demographics table ---
silver_demographics = silver_income[['GEOID','median_household_income']].merge(
    silver_poverty[['GEOID','pct_poverty']], on='GEOID', how='outer'
)

print(f'\n📊 Demographics built: {len(silver_demographics)} tracts')
print(silver_demographics.describe().round(2))
print('\n✅ Census silver layer complete')

SILVER LAYER — Transforming Census Data
  Income column found: ['S1901_C01_012E']
  Poverty column found: ['S1701_C03_001E']

  B08201 rows after transpose: 534
  B08201 sample columns: ['GEO_ID', '\xa0\xa0\xa0\xa0No vehicle available', '\xa0\xa0\xa0\xa01 vehicle available', '\xa0\xa0\xa0\xa02 vehicles available']
  Total col: []
  Zero vehicle col: ['\xa0\xa0\xa0\xa0No vehicle available', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0No vehicle available', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0No vehicle available', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0No vehicle available', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0No vehicle available']

  Total pop col: []
  Black pop col: ['\xa0\xa0\xa0\xa0Black or African American alone']

📊 Demographics built: 267 tracts
       median_household_income  pct_poverty
count                   264.00       265.00
mean                  85786.45        13.57
std                   36034.89         9.22
min                   14820.00         0.60
25%                   58770.50         

## 11 - SILVER LAYER: Join Census Demographics to Stops via Census Tracts

In [30]:
print('=' * 55)
print('SILVER LAYER — Joining Census Data to Stops')
print('=' * 55)

# Reproject tracts to WGS84
silver_tracts = bronze_tracts.copy()
if silver_tracts.crs != 'EPSG:4326':
    silver_tracts = silver_tracts.to_crs('EPSG:4326')

# Filter to Orange County (FIPS county code 095)
silver_tracts = silver_tracts[silver_tracts['COUNTYFP'] == '095'].copy()

# Build GEOID
silver_tracts['GEOID'] = silver_tracts['STATEFP'] + silver_tracts['COUNTYFP'] + silver_tracts['TRACTCE']

# FIX: Drop leftover index_right column from previous spatial join
silver_stops_clean = silver_stops_with_neighborhood.copy()
silver_stops_clean = silver_stops_clean.drop(
    columns=[c for c in silver_stops_clean.columns if 'index_right' in c],
    errors='ignore'
)

# Spatial join: stops → tracts
print('\n⏳ Joining stops to census tracts...')
stops_with_tracts = gpd.sjoin(
    silver_stops_clean,
    silver_tracts[['geometry', 'GEOID']],
    how='left',
    predicate='within'
)

# Join demographics to stops
silver_stops_final = stops_with_tracts.merge(
    silver_demographics,
    on='GEOID',
    how='left'
)

# Drop any remaining index_right columns
silver_stops_final = silver_stops_final.drop(
    columns=[c for c in silver_stops_final.columns if 'index_right' in c],
    errors='ignore'
)

print(f'\n📊 Final Silver Stops — Sample:')
display(silver_stops_final[[
    'stop_id','stop_name','service_type','neighborhood_name',
    'is_equity_zone','median_household_income'
]].head(5))

print('\n✅ Demographics joined to stops')

SILVER LAYER — Joining Census Data to Stops

⏳ Joining stops to census tracts...

📊 Final Silver Stops — Sample:


,stop_id,stop_name,service_type,neighborhood_name,is_equity_zone,median_household_income
0,175,ROSEMONT SUPERSTOP,Standard Link,Orlando,False,48281.00
1,601,N JOHN YOUNG PKWY & LYNDELL DR,Standard Link,Unclassified,False,NaN
2,615,N JOHN YOUNG PKWY & REGATTA BAY BLVD,Standard Link,Unclassified,False,NaN
3,616,DENN JOHN LN & SUNBURST WAY,Standard Link,Unclassified,False,NaN
4,617,VALENCIA COLLEGE-OSCEOLA CAMPUS,Standard Link,Unclassified,False,NaN



✅ Demographics joined to stops


## 12 - SILVER LAYER: Build Master Service Events Table

In [34]:
# FIX: Force trip_id to string in both tables before merging
silver_stop_times['trip_id'] = silver_stop_times['trip_id'].astype(str).str.strip()
silver_trips['trip_id']      = silver_trips['trip_id'].astype(str).str.strip()
silver_trips['route_id']     = silver_trips['route_id'].astype(str).str.strip()
silver_routes['route_id']    = silver_routes['route_id'].astype(str).str.strip()

# Step 1: stop_times + trips
events = silver_stop_times.merge(
    silver_trips[['trip_id','route_id','service_id','is_weekday','is_weekend']],
    on='trip_id',
    how='left'
)

# Step 2: + routes
events = events.merge(
    silver_routes[['route_id','route_short_name','route_long_name','service_level']],
    on='route_id',
    how='left'
)

# Step 3: + stops (with neighborhood + demographics)
stop_attributes = silver_stops_final[[
    'stop_id','stop_name','stop_lat','stop_lon',
    'service_type','neighborhood_name','is_equity_zone','ada_accessible',
    'median_household_income','GEOID'
]].drop_duplicates(subset='stop_id')

# FIX: Force stop_id to string in both
events['stop_id']                = events['stop_id'].astype(str).str.strip()
stop_attributes = stop_attributes.copy()
stop_attributes['stop_id']       = stop_attributes['stop_id'].astype(str).str.strip()

events = events.merge(stop_attributes, on='stop_id', how='left')

silver_master_events = events.copy()
print(f'\n📊 Master Events Table Summary:')
print(f'   Total service events:   {len(silver_master_events):,}')
print(f'   Unique stops:           {silver_master_events["stop_id"].nunique():,}')
print(f'   Unique routes:          {silver_master_events["route_id"].nunique():,}')
print('\n✅ Master events table built')


📊 Master Events Table Summary:
   Total service events:   74,938
   Unique stops:           275
   Unique routes:          62

✅ Master events table built


## 13 - Save Silver Layer Outputs

In [35]:
# ============================================================
# Save all Silver Layer outputs for use in modeling stage
# ============================================================

print('=' * 55)
print('Saving Silver Layer Outputs')
print('=' * 55)

SILVER_PATH = os.path.join(BASE_PATH, 'silver/')
os.makedirs(SILVER_PATH, exist_ok=True)

# Save as CSV (drop geometry column for non-geo files)
silver_master_events.drop(columns='geometry', errors='ignore').to_csv(
    os.path.join(SILVER_PATH, 'silver_transit_events.csv'), index=False
)
print(f'  ✅ silver_transit_events.csv  — {len(silver_master_events):,} rows')

silver_stops_final.drop(columns='geometry', errors='ignore').to_csv(
    os.path.join(SILVER_PATH, 'silver_stops.csv'), index=False
)
print(f'  ✅ silver_stops.csv           — {len(silver_stops_final):,} rows')

silver_routes.to_csv(
    os.path.join(SILVER_PATH, 'silver_routes.csv'), index=False
)
print(f'  ✅ silver_routes.csv          — {len(silver_routes):,} rows')

silver_demographics.to_csv(
    os.path.join(SILVER_PATH, 'silver_demographics.csv'), index=False
)
print(f'  ✅ silver_demographics.csv    — {len(silver_demographics):,} rows')

# Save GeoJSON for mapping
silver_stops_final.to_file(
    os.path.join(SILVER_PATH, 'silver_stops.geojson'), driver='GeoJSON'
)
print(f'  ✅ silver_stops.geojson       — spatial file saved')

print(f'\n📁 All silver files saved to: {SILVER_PATH}')
print('\n🎉 ETL Pipeline Complete — Ready for Star Schema Modeling')

Saving Silver Layer Outputs
  ✅ silver_transit_events.csv  — 74,938 rows
  ✅ silver_stops.csv           — 3,532 rows
  ✅ silver_routes.csv          — 62 rows
  ✅ silver_demographics.csv    — 267 rows
  ✅ silver_stops.geojson       — spatial file saved

📁 All silver files saved to: /content/drive/MyDrive/Transit Equity Project/silver/

🎉 ETL Pipeline Complete — Ready for Star Schema Modeling


## 14 - Data Quality Report

In [36]:
# ============================================================
# Final Data Quality Report — document for your PDF report
# ============================================================

print('=' * 60)
print('DATA QUALITY REPORT — Transit Equity ETL Pipeline')
print('=' * 60)

print('''
BRONZE LAYER (Raw Ingestion)
------------------------------------------------------------
Source 1: LYNX GTFS
  - stops.txt, stop_times.txt, routes.txt, trips.txt, calendar.txt
  - Loaded as-is with no transformations
  - Documented row counts and null values per file

Source 2: Orange County GIS (GeoJSON)
  - Neighborhood/Zoning boundary polygons
  - CRS verified on load

Source 3: Census ACS 5-Year 2024
  - 6 tables: S0801, S1901, S1701, B08201, B02001, B03002
  - Loaded with skiprows=[1] to handle double header rows

TRANSFORMATION RULES APPLIED (Silver Layer)
------------------------------------------------------------
Rule 1: Dropped stops with null coordinates
Rule 2: Validated stop coordinates within Orange County bounds
Rule 3: Classified service type (LYMMO / NeighborLink / Standard)
Rule 4: Converted GTFS time strings to minutes (handling >24:00:00)
Rule 5: Classified Peak / Off-Peak / Late Night time periods
Rule 6: Calculated headway per stop; flagged >120 min as null
Rule 7: Spatial join — tagged stops with neighborhood name
Rule 8: Applied equity_zone flag for Pine Hills / Parramore
Rule 9: Joined census tracts to stops for demographic overlay
Rule 10: Merged all tables into master service events table

KNOWN LIMITATIONS
------------------------------------------------------------
- GTFS represents SCHEDULED times, not real-time arrivals
- Shelter/lighting amenity data not in GTFS — requires manual audit
- Census tract boundaries do not perfectly align with neighborhood names
- Stops outside OC zoning polygons tagged as 'Unclassified'
''')

print('=' * 60)
print('END OF ETL PIPELINE')
print('=' * 60)

DATA QUALITY REPORT — Transit Equity ETL Pipeline

BRONZE LAYER (Raw Ingestion)
------------------------------------------------------------
Source 1: LYNX GTFS
  - stops.txt, stop_times.txt, routes.txt, trips.txt, calendar.txt
  - Loaded as-is with no transformations
  - Documented row counts and null values per file

Source 2: Orange County GIS (GeoJSON)
  - Neighborhood/Zoning boundary polygons
  - CRS verified on load

Source 3: Census ACS 5-Year 2024
  - 6 tables: S0801, S1901, S1701, B08201, B02001, B03002
  - Loaded with skiprows=[1] to handle double header rows

TRANSFORMATION RULES APPLIED (Silver Layer)
------------------------------------------------------------
Rule 1: Dropped stops with null coordinates
Rule 2: Validated stop coordinates within Orange County bounds
Rule 3: Classified service type (LYMMO / NeighborLink / Standard)
Rule 4: Converted GTFS time strings to minutes (handling >24:00:00)
Rule 5: Classified Peak / Off-Peak / Late Night time periods
Rule 6: Calculat